In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

# 1. Load the dataset
# We will do a quick re-encode here just in case you didn't save the Level 2 CSV
df = pd.read_csv('../data/Dataset.csv')
df['Has Table Booking (Encoded)'] = df['Has Table booking'].map({'Yes': 1, 'No': 0})
df['Has Online Delivery (Encoded)'] = df['Has Online delivery'].map({'Yes': 1, 'No': 0})
df['Restaurant Name Length'] = df['Restaurant Name'].astype(str).apply(len)
df['Address Length'] = df['Address'].astype(str).apply(len)

print("--- Level 3, Task 1: Predictive Modeling ---\n")

# 2. Select features and target variable
features = ['Average Cost for two', 'Has Table Booking (Encoded)', 'Has Online Delivery (Encoded)', 
            'Price range', 'Votes', 'Restaurant Name Length', 'Address Length']
X = df[features].copy()
y = df['Aggregate rating']

# 3. Split the dataset into training (80%) and testing (20%) sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 4. Initialize the models
models = {
    "Linear Regression": LinearRegression(),
    "Decision Tree": DecisionTreeRegressor(random_state=42),
    "Random Forest": RandomForestRegressor(n_estimators=100, random_state=42)
}

# 5. Train, predict, and evaluate each model
print("--- Model Evaluation Results ---")
for name, model in models.items():
    model.fit(X_train, y_train)          
    y_pred = model.predict(X_test)       
    
    # Calculate metrics
    r2 = r2_score(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    
    print(f"{name}:")
    print(f"  R-squared (Accuracy score): {r2:.4f}")
    print(f"  Mean Squared Error: {mse:.4f}\n")

# 6. Display the feature importances from the Random Forest model
rf_model = models["Random Forest"]
importances = pd.Series(rf_model.feature_importances_, index=X.columns).sort_values(ascending=False)
print("--- Feature Importance (What drives the rating?) ---")
print(importances.round(4))

--- Level 3, Task 1: Predictive Modeling ---

--- Model Evaluation Results ---
Linear Regression:
  R-squared (Accuracy score): 0.2630
  Mean Squared Error: 1.6775

Decision Tree:
  R-squared (Accuracy score): 0.9060
  Mean Squared Error: 0.2141

Random Forest:
  R-squared (Accuracy score): 0.9511
  Mean Squared Error: 0.1112

--- Feature Importance (What drives the rating?) ---
Votes                            0.9544
Address Length                   0.0138
Average Cost for two             0.0133
Restaurant Name Length           0.0102
Price range                      0.0044
Has Online Delivery (Encoded)    0.0026
Has Table Booking (Encoded)      0.0013
dtype: float64


In [3]:
import pandas as pd
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go

# 1. Load and clean the dataset
df = pd.read_csv('../data/Dataset.csv')
df_clean = df.dropna(subset=['Cuisines']).copy()

print("--- Level 3, Task 2: Ultra-Clear Customer Preference Analysis ---\n")

# --- PREPARE DATA 1: Most Popular (Most Voted) ---
cuisine_votes = df_clean.groupby('Cuisines')['Votes'].sum().reset_index()
top_10_popular = cuisine_votes.sort_values(by='Votes', ascending=False).head(10)

# --- PREPARE DATA 2: Highest Rated (Best Reviewed with Min. 1000 Votes) ---
cuisine_ratings = df_clean.groupby('Cuisines').agg({'Aggregate rating': 'mean', 'Votes': 'sum'}).reset_index()
popular_cuisines = cuisine_ratings[cuisine_ratings['Votes'] > 1000]
top_10_rated = popular_cuisines.sort_values(by='Aggregate rating', ascending=False).head(10)
top_10_rated['Aggregate rating'] = top_10_rated['Aggregate rating'].round(2)


# 2. Create a Side-by-Side Super Dashboard
fig = make_subplots(
    rows=1, cols=2, 
    subplot_titles=(
        "🔥 1. MOST POPULAR CUISINES<br><sub>Ordered the most by total count</sub>", 
        "⭐ 2. HIGHEST RATED CUISINES<br><sub>Consistently gets the best reviews (Out of 5)</sub>"
    ),
    horizontal_spacing=0.15
)

# Add Left Chart: Popularity Bars
fig.add_trace(
    go.Bar(
        x=top_10_popular['Cuisines'],
        y=top_10_popular['Votes'],
        text=top_10_popular['Votes'].apply(lambda x: f"{x:,} votes"),
        textposition='outside',
        marker=dict(color=px.colors.qualitative.Vivid[:10]), # Vibrant, distinct colors
        name="Popularity"
    ),
    row=1, col=1
)

# Add Right Chart: Rating Bars
fig.add_trace(
    go.Bar(
        x=top_10_rated['Cuisines'],
        y=top_10_rated['Aggregate rating'],
        text=top_10_rated['Aggregate rating'].apply(lambda x: f"{x} ⭐"),
        textposition='outside',
        marker=dict(color=px.colors.qualitative.Pastel[:10]), # Distinct contrasting colors
        name="Rating"
    ),
    row=1, col=2
)

# 3. Polish the layout for maximum clarity (Large fonts, bold text)
fig.update_layout(
    title_text="🍕 WHAT DO CUSTOMERS ACTUALLY PREFER? POPULARITY VS. QUALITY",
    title_font_size=22,
    title_font_family="Arial",
    showlegend=False,
    height=600,
    margin=dict(t=120, b=120, l=50, r=50)
)

# Adjust axes labels and spacing so cuisine names don't overlap
fig.update_xaxes(tickangle=45, tickfont=dict(size=11, family="Arial", color="black"))
fig.update_yaxes(title_text="Total Orders / Votes", row=1, col=1)
fig.update_yaxes(title_text="Average Star Rating", range=[3.5, 5.2], row=1, col=2)

fig.show()

--- Level 3, Task 2: Ultra-Clear Customer Preference Analysis ---



In [4]:
import pandas as pd

# Load the dataset
df = pd.read_csv('../data/Dataset.csv')
df_clean = df.dropna(subset=['Cuisines']).copy()

print("--- Answering: Do specific cuisines tend to receive higher ratings? ---\n")

# Calculate average rating and total votes for each cuisine
cuisine_analysis = df_clean.groupby('Cuisines').agg(
    Average_Rating=('Aggregate rating', 'mean'),
    Total_Votes=('Votes', 'sum')
).reset_index()

# Filter for "Proven Winners" (High rating AND a reliable amount of votes to prove it's not a fluke)
# Let's set the bar at a minimum of 500 votes and a rating of 4.5 or higher
elite_cuisines = cuisine_analysis[
    (cuisine_analysis['Total_Votes'] >= 500) & 
    (cuisine_analysis['Average_Rating'] >= 4.5)
].sort_values(by='Average_Rating', ascending=False)

print("Yes! The data determines that the following specific cuisines tend to receive significantly higher ratings (4.5+):\n")

# Formatting the output to look like a clean report
for index, row in elite_cuisines.iterrows():
    print(f"⭐ {row['Average_Rating']:.2f} Rating | {row['Cuisines']} ({row['Total_Votes']:,} votes)")

--- Answering: Do specific cuisines tend to receive higher ratings? ---

Yes! The data determines that the following specific cuisines tend to receive significantly higher ratings (4.5+):

⭐ 4.90 Rating | American, Coffee and Tea (570 votes)
⭐ 4.90 Rating | American, Caribbean, Seafood (548 votes)
⭐ 4.90 Rating | American, BBQ, Sandwich (1,252 votes)
⭐ 4.90 Rating | American, Sandwich, Tea (1,457 votes)
⭐ 4.90 Rating | European, Asian, Indian (621 votes)
⭐ 4.90 Rating | European, German (1,413 votes)
⭐ 4.90 Rating | Burger, Bar Food, Steak (2,238 votes)
⭐ 4.90 Rating | Hawaiian, Seafood (1,343 votes)
⭐ 4.90 Rating | Italian, Deli (1,424 votes)
⭐ 4.90 Rating | Sunda, Indonesian (5,514 votes)
⭐ 4.90 Rating | Mughlai, Lucknowi (1,057 votes)
⭐ 4.90 Rating | Continental, Indian (641 votes)
⭐ 4.85 Rating | Filipino, Mexican (1,364 votes)
⭐ 4.80 Rating | Indian, Continental (2,510 votes)
⭐ 4.80 Rating | Italian, American, Pizza (10,934 votes)
⭐ 4.80 Rating | Contemporary, Italian (542 votes)


In [5]:
import pandas as pd
import plotly.express as px

# 1. Load and clean the dataset
df = pd.read_csv('../data/Dataset.csv')
df_clean = df.dropna(subset=['Cuisines']).copy()

# 2. Calculate average rating and total votes for each cuisine
cuisine_analysis = df_clean.groupby('Cuisines').agg(
    Average_Rating=('Aggregate rating', 'mean'),
    Total_Votes=('Votes', 'sum')
).reset_index()

# 3. Filter for "Elite Cuisines" (Min 500 votes AND 4.5+ Rating)
elite_cuisines = cuisine_analysis[
    (cuisine_analysis['Total_Votes'] >= 500) & 
    (cuisine_analysis['Average_Rating'] >= 4.5)
].copy()

# Sort by rating first, then by votes to break any ties
elite_cuisines = elite_cuisines.sort_values(by=['Average_Rating', 'Total_Votes'], ascending=[True, True])
elite_cuisines['Average_Rating'] = elite_cuisines['Average_Rating'].round(2)

# 4. Build the Visual Chart
fig = px.bar(
    elite_cuisines,
    x='Average_Rating',
    y='Cuisines',
    orientation='h',
    text=elite_cuisines.apply(lambda row: f"{row['Average_Rating']} ⭐ ({row['Total_Votes']:,} votes)", axis=1),
    color='Average_Rating',
    color_continuous_scale='Greens', # Using a green scale to signify "Excellent"
    title='🏆 The "Elite" Cuisines (4.5+ Rating & 500+ Votes)'
)

# 5. Format layout for ultra-clarity
fig.update_traces(textposition='inside', textfont_size=13, textfont_color='black')
fig.update_layout(
    xaxis_title="Average Rating (Out of 5.0)",
    yaxis_title="Cuisine Type",
    xaxis=dict(range=[4.4, 5.0]), # Zoom in on the high ratings so differences are visible
    coloraxis_showscale=False,
    height=500,
    margin=dict(l=200, r=20, t=60, b=40)
)

fig.show()

In [7]:
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# 1. Load the dataset
df = pd.read_csv('../data/Dataset.csv')

print("--- Level 3, Task 3 (Part 1): Ultra-Clear Rating Distribution ---\n")

# --- PREPARE DATA ---
# Left Chart: We filter out '0.0' so people can actually see the curve of the rated restaurants
rated_restaurants = df[df['Aggregate rating'] > 0]

# Right Chart: We count the official Zomato rating categories
category_counts = df['Rating text'].value_counts().reindex(
    ['Not rated', 'Poor', 'Average', 'Good', 'Very Good', 'Excellent']
).reset_index()
category_counts.columns = ['Category', 'Count']

# Define intuitive "Traffic Light" colors for the categories
color_map = {
    'Not rated': '#B0BEC5',  # Grey
    'Poor': '#EF5350',       # Red
    'Average': '#FFCA28',    # Yellow
    'Good': '#9CCC65',       # Light Green
    'Very Good': '#66BB6A',  # Green
    'Excellent': '#2E7D32'   # Dark Green
}

# 2. Create a Side-by-Side Dashboard
fig = make_subplots(
    rows=1, cols=2, 
    subplot_titles=(
        "📈 1. THE SCORE CURVE<br><sub>Where do rated restaurants usually land?</sub>", 
        "🏷️ 2. OFFICIAL CATEGORIES<br><sub>How Zomato officially labels them</sub>"
    ),
    horizontal_spacing=0.12
)

# Add Left Chart: The Rating Curve (Histogram)
fig.add_trace(
    go.Histogram(
        x=rated_restaurants['Aggregate rating'],
        nbinsx=25,
        marker_color='#42A5F5', # Friendly Blue
        name="Scores"
    ),
    row=1, col=1
)

# Add Right Chart: Official Categories (Bar Chart)
fig.add_trace(
    go.Bar(
        x=category_counts['Category'],
        y=category_counts['Count'],
        text=category_counts['Count'].apply(lambda x: f"{x:,}"), # Adds exact numbers with commas
        textposition='outside',
        marker_color=[color_map[cat] for cat in category_counts['Category']],
        name="Categories"
    ),
    row=1, col=2
)

# 3. Polish the layout so it is impossible to misunderstand
fig.update_layout(
    title_text="⭐ HOW ARE RESTAURANTS ACTUALLY RATED?",
    title_font_size=22,
    showlegend=False,
    height=550,
    margin=dict(t=120, b=80, l=50, r=50),
    plot_bgcolor='rgba(245, 245, 245, 1)' # Soft grey background makes colors pop
)

# Make the axes plain English
fig.update_xaxes(title_text="Star Rating (1.0 to 5.0)", row=1, col=1)
fig.update_yaxes(title_text="Number of Restaurants", row=1, col=1)

fig.update_xaxes(title_text="Zomato Label", row=1, col=2)
fig.update_yaxes(title_text="Total Count", range=[0, 4500], row=1, col=2)

fig.show()

--- Level 3, Task 3 (Part 1): Ultra-Clear Rating Distribution ---



In [8]:
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# 1. Load the dataset
df = pd.read_csv('../data/Dataset.csv')
df_clean = df.dropna(subset=['Cuisines']).copy()

print("--- Level 3, Task 3 (Part 2): Comparing Cities and Cuisines ---\n")

# --- PREPARE DATA 1: Top 10 Cities by Average Rating ---
# We grab the 10 cities with the most restaurants to ensure a fair comparison
top_cities_list = df['City'].value_counts().head(10).index
city_ratings = df[df['City'].isin(top_cities_list)].groupby('City')['Aggregate rating'].mean().reset_index()
city_ratings = city_ratings.sort_values(by='Aggregate rating', ascending=True)
city_ratings['Aggregate rating'] = city_ratings['Aggregate rating'].round(2)

# --- PREPARE DATA 2: Top 10 Cuisines by Average Rating ---
# We grab the 10 most frequent cuisines for a fair comparison
top_cuisines_list = df_clean['Cuisines'].value_counts().head(10).index
cuisine_ratings = df_clean[df_clean['Cuisines'].isin(top_cuisines_list)].groupby('Cuisines')['Aggregate rating'].mean().reset_index()
cuisine_ratings = cuisine_ratings.sort_values(by='Aggregate rating', ascending=True)
cuisine_ratings['Aggregate rating'] = cuisine_ratings['Aggregate rating'].round(2)


# 2. Create the Side-by-Side Dashboard
fig = make_subplots(
    rows=1, cols=2, 
    subplot_titles=(
        "🏙️ 1. TOP CITIES<br><sub>Average rating of major food hubs</sub>", 
        "🍝 2. TOP CUISINES<br><sub>Average rating of the most common foods</sub>"
    ),
    horizontal_spacing=0.15
)

# Add Left Chart: Cities
fig.add_trace(
    go.Bar(
        x=city_ratings['Aggregate rating'],
        y=city_ratings['City'],
        orientation='h',
        text=city_ratings['Aggregate rating'].apply(lambda x: f"{x} ⭐"),
        textposition='outside',
        marker_color='#FFA726', # Warm Orange
        name="Cities"
    ),
    row=1, col=1
)

# Add Right Chart: Cuisines
fig.add_trace(
    go.Bar(
        x=cuisine_ratings['Aggregate rating'],
        y=cuisine_ratings['Cuisines'],
        orientation='h',
        text=cuisine_ratings['Aggregate rating'].apply(lambda x: f"{x} ⭐"),
        textposition='outside',
        marker_color='#26A69A', # Cool Teal
        name="Cuisines"
    ),
    row=1, col=2
)

# 3. Polish the layout
fig.update_layout(
    title_text="🌍 WHICH CITIES & CUISINES HAVE THE BEST FOOD?",
    title_font_size=22,
    showlegend=False,
    height=550,
    margin=dict(t=120, b=80, l=50, r=50),
    plot_bgcolor='rgba(245, 245, 245, 1)'
)

# Zoom in the X-axis so the differences are super obvious
fig.update_xaxes(title_text="Average Star Rating", range=[2.0, 4.5], row=1, col=1)
fig.update_xaxes(title_text="Average Star Rating", range=[2.0, 4.5], row=1, col=2)

fig.show()

--- Level 3, Task 3 (Part 2): Comparing Cities and Cuisines ---



In [9]:
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# 1. Load the dataset
df = pd.read_csv('../data/Dataset.csv')

print("--- Level 3, Task 3 (Part 3): Feature Relationships ---\n")

# --- PREPARE THE DATA ---
# Feature 1: Price Range vs Rating
price_rating = df.groupby('Price range')['Aggregate rating'].mean().reset_index()
price_rating['Price Label'] = price_rating['Price range'].map({
    1: '1 (Cheapest)', 2: '2 (Moderate)', 3: '3 (Expensive)', 4: '4 (Luxury)'
})
price_rating['Aggregate rating'] = price_rating['Aggregate rating'].round(2)

# Feature 2: Table Booking vs Rating
book_rating = df.groupby('Has Table booking')['Aggregate rating'].mean().reset_index()
book_rating['Aggregate rating'] = book_rating['Aggregate rating'].round(2)

# Feature 3: Online Delivery vs Rating
del_rating = df.groupby('Has Online delivery')['Aggregate rating'].mean().reset_index()
del_rating['Aggregate rating'] = del_rating['Aggregate rating'].round(2)


# 2. CREATE THE 3-PART DASHBOARD
fig = make_subplots(
    rows=1, cols=3, 
    subplot_titles=(
        "💰 1. PRICE IMPACT<br><sub>Does expensive mean better?</sub>", 
        "📅 2. TABLE BOOKING<br><sub>Do reservations matter?</sub>",
        "🛵 3. ONLINE DELIVERY<br><sub>Does delivery affect scores?</sub>"
    )
)

# Add Chart 1: Price 
fig.add_trace(
    go.Bar(
        x=price_rating['Price Label'],
        y=price_rating['Aggregate rating'],
        text=price_rating['Aggregate rating'].apply(lambda x: f"{x} ⭐"),
        textposition='outside',
        marker_color=['#90CAF9', '#64B5F6', '#2196F3', '#1565C0'], # Light to Dark Blue
        name="Price"
    ), row=1, col=1
)

# Add Chart 2: Table Booking
fig.add_trace(
    go.Bar(
        x=book_rating['Has Table booking'].replace({'Yes': 'Offers Booking', 'No': 'No Booking'}),
        y=book_rating['Aggregate rating'],
        text=book_rating['Aggregate rating'].apply(lambda x: f"{x} ⭐"),
        textposition='outside',
        marker_color=['#EF5350', '#66BB6A'], # Red for No, Green for Yes
        name="Booking"
    ), row=1, col=2
)

# Add Chart 3: Online Delivery
fig.add_trace(
    go.Bar(
        x=del_rating['Has Online delivery'].replace({'Yes': 'Offers Delivery', 'No': 'No Delivery'}),
        y=del_rating['Aggregate rating'],
        text=del_rating['Aggregate rating'].apply(lambda x: f"{x} ⭐"),
        textposition='outside',
        marker_color=['#EF5350', '#66BB6A'], # Red for No, Green for Yes
        name="Delivery"
    ), row=1, col=3
)

# 3. POLISH THE LAYOUT FOR MAXIMUM CLARITY
fig.update_layout(
    title_text="🔍 WHAT DRIVES A HIGHER RATING? (Features vs. Ratings)",
    title_font_size=22,
    showlegend=False,
    height=550,
    plot_bgcolor='rgba(245, 245, 245, 1)',
    margin=dict(t=120, b=50, l=50, r=50)
)

# Lock the Y-Axis across all 3 charts so the visual comparison is perfectly fair
fig.update_yaxes(range=[2.0, 4.5], title_text="Average Star Rating", row=1, col=1)
fig.update_yaxes(range=[2.0, 4.5], row=1, col=2)
fig.update_yaxes(range=[2.0, 4.5], row=1, col=3)

fig.show()

--- Level 3, Task 3 (Part 3): Feature Relationships ---

